In [ ]:

from tensorflow.keras import layers, models, backend as K




from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Conv2DTranspose, Cropping2D, UpSampling2D, Reshape

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.inception_v3 import InceptionV3

from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential

from tensorflow.keras.applications import ResNet50, InceptionResNetV2
from tensorflow.keras import Input
from tensorflow import keras

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import tensorflow as tf


from keras.layers import Lambda, Input



In [ ]:
def extract_features(img_path):
    img = image.load_img(img_path, target_size=(400, 320))
    x = image.img_to_array(img)
    x = x /255.0
    return x

In [ ]:
input_shape = (400,320, 3)
datagen = ImageDataGenerator(rescale=1./255)
    
path_train = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/train"
path_val = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/validation"
    
batch_size = 32
target_size = (input_shape[0],input_shape[1])
train_generator = datagen.flow_from_directory(path_train, target_size=target_size,  batch_size=batch_size, class_mode='input', shuffle=True, color_mode='rgb')

validation_generator = datagen.flow_from_directory(path_val, target_size=target_size , batch_size=batch_size, class_mode='input', shuffle=True, color_mode='rgb')

In [ ]:
latent_dims = 512

In [ ]:


# Assuming 'input_shape' is defined elsewhere in your code.
input_img = Input(shape=input_shape) 

x = Conv2D(128, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
conv_shape = K.int_shape(x)
x = Flatten()(x)

z_mu = Dense(512, activation='relu')(x)
z_sigma = Dense(512, activation='relu')(x)





In [ ]:
# This function will sample from the latent distribution
#def sampling(args):
 #   z_mean, z_log_var = args
  #  batch = K.shape(z_mean)[0]
   # dim = K.int_shape(z_mean)[1]
    #epsilon = K.random_normal(shape=(batch, dim))
    #return z_mean + K.exp(0.5 * z_log_var) * epsilon

#z = Lambda(sampling)([z_mean, z_log_var])

def sample_z(args):
    z_mu, z_sigma = args
    eps = K.random_normal(shape=(K.shape(z_mu)[0], K.int_shape(z_mu)[1]))
    return z_mu + K.exp(z_sigma/2) * eps

z = Lambda(sample_z, output_shape=(latent_dims,), name='z')([z_mu,z_sigma])
encoder = Model(input_img, [z_mu, z_sigma,z], name='encoder')

In [ ]:
encoder.summary()

In [ ]:
target_shape = (8, 10, 64)
# Reshape to the dimensions before the first Dense layer in the decoder
decoder_input = Input(shape=latent_dims, name='decoder_input')
x = Dense(target_shape[0] * target_shape[1] * target_shape[2], activation='relu')(decoder_input)
x = Reshape(target_shape)(x)

# Decoder layers
x = Conv2DTranspose(256, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2DTranspose(16, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
x = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)
x = Cropping2D(cropping=((56,56), (160,160)))(x)
decoder = Model(decoder_input, x, name='decoder')
decoder.summary()



In [ ]:
z_decoded = decoder(z)

In [ ]:
class CustomLayer(keras.layers.Layer):
    def vae_loss(self, x, z_decoded):
        x = K.flatten(x)
        z_decoded = K.flatten(z_decoded)
        
        # Compute binary crossentropy directly with TensorFlow
        recon_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(x, z_decoded))
        
        # Ensure kl_divergence_loss is calculated with TensorFlow operations
        kl_divergence_loss = -5e-4 * K.mean(1 + z_sigma - K.square(z_mu) - K.exp(z_sigma), axis=-1)
        
        # Sum the two components of the loss
        return recon_loss + kl_divergence_loss
    
    def call(self, inputs):
        x, z_decoded = inputs
        loss = self.vae_loss(x, z_decoded)
        
        # Properly add loss to the layer
        self.add_loss(loss)
        return x
    
y = CustomLayer()([input_img, z_decoded])
        
        
        

In [ ]:
vae = Model(input_img, y, name='vae')
vae.compile(optimizer='adam', loss=None)
vae.summary()

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import cv2
def extract_features(img_path):
    img = image.load_img(img_path, target_size=(400, 320),color_mode = "rgb")
    x = image.img_to_array(img)
    x = x / 255.0
    return x
import glob, os
 
xtrain = []

path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):

        if label == "Inlet (5 marking)":
            xtrain.append(extract_features(image_path))

print('Done')
            
xtrain = np.array(xtrain) 

In [ ]:
epochs = 2000
steps_per_epoch = len(train_generator)

vae.fit(xtrain, steps_per_epoch=steps_per_epoch, epochs=epochs, batch_size=batch_size)